# Clasificación de la expresión a partir del clip

Tercera etapa. Toma los clips de 16 frames alineados que produce `02_roi_y_normalizacion.ipynb` y
entrena un clasificador de tres clases: `Disgustado`, `Interesado`, `Sereno`.

Cubre el **objetivo específico 3** del plan de trabajo.

## Qué hay que poder leer en este notebook

El entregable no es sólo un número. Es el número **más** el protocolo que permite creerle:

1. **La partición es por ejemplar**, nunca por clip. 10 de los 73 caballos tienen clips íntegros en
   más de una clase; particionar por clip los pondría de los dos lados del corte. Eso no da error:
   da una métrica optimista y falsa. **Cuánto de optimista está medido en el Paso 5: F1 macro 0,53
   sin mirar un solo píxel.**
2. **Tres líneas de base declaradas antes de entrenar**, y el criterio de aceptación anclado en
   ellas. Sin eso, un 0,60 puede ser excelente o irrelevante y no hay forma de saber cuál.
3. **El resultado se informa aunque no cumpla el criterio.** No lo cumple.

## El resultado, por adelantado

**La hipótesis del trabajo no queda sostenida por esta evidencia.** El mejor modelo de las 18
configuraciones probadas es **invariante al orden de los frames**: barajar el clip no cambia una
predicción. Se probaron tres cabezas temporales en dos regímenes de capacidad y con las muestras
igualadas, y en ninguna el orden aportó.

Lo que sí quedó demostrado es que **las imágenes tienen señal**: de 0,188 (clase mayoritaria) a
0,626 con partición honesta. Lo que faltaría para más son **ejemplares**, no arquitectura — y eso
también está medido, no supuesto.

## Paso 0 — Arranque: montar Drive y cargar el paquete `equinos`

In [ ]:
# ============================================================
# ARRANQUE — persistencia en Google Drive
# ============================================================
# Autocontenida a propósito: corre ANTES de que el paquete `equinos` esté disponible.

import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODIGO = Path("/content/drive/MyDrive") / NOMBRE_PROYECTO / "codigo"
    if CODIGO.is_dir():
        sys.path.insert(0, str(CODIGO))
else:
    sys.path.insert(0, str(Path.cwd()))

from equinos import caracteristicas, clasificacion, entorno, particion, rutas

RAIZ = rutas.inicializar()

## Paso 0a — Parámetros de la etapa

In [ ]:
# Parámetros de la clasificación.
#
# Papermill inyecta una celda justo debajo de ésta cuando el notebook se ejecuta desde
# 00_pipeline.ipynb. Ejecutado a mano, valen estos valores por defecto.

# Folds de la validación cruzada. 4 y no 5: con 5 hay folds con apenas 2 clips de `Disgustado` en
# test, y una métrica calculada sobre dos muestras no dice nada.
FOLDS = 4

# Vistas aumentadas por clip. Se midió que la aumentación NO mejora nada (-0,005 a +0,000), así
# que 0 la desactiva y es el valor por defecto. Se deja el parámetro porque la medición es parte
# del resultado y hay que poder rehacerla.
VISTAS_AUMENTADAS = 0

# True vuelve a extraer las características aunque la caché esté vigente.
REEXTRAER = False

# True cuando el notebook lo ejecuta 00_pipeline.ipynb: apaga las salidas largas.
EJECUCION_AUTOMATICA = False

## Paso 1 — Verificar el entorno

La etapa 03 **no** necesita DeepLabCut: sólo `torch`, `torchvision` y `scikit-learn`. Es la única
del pipeline que no arrastra la restricción de `numpy<2`.

In [ ]:
entorno.verificar("modelado", gpu=True)

# Bloque 1: El conjunto de clips y la partición

## Paso 2 — Clips íntegros y partición por ejemplar

**La unidad es el clip íntegro**: sus 16 frames alineados. Un clip a medio alinear mezcla recortes
de keypoints con recortes de caja dentro de la misma muestra, y a qué frame le toca cuál
correlaciona con el ejemplar, que predice la clase.

La partición se **persiste**. Todas las corridas —líneas de base, variantes de cabeza, ablación—
leen los mismos cortes; comparar números medidos sobre cortes distintos no dice nada.

In [ ]:
clips = particion.informe()

## Paso 3 — Verificar que el orden temporal es el que declara el CSV

Un desorden temporal no rompe nada: no lanza excepciones y todos los números siguen saliendo. Pero
**arruinaría la conclusión central de este notebook**: si los frames llegaran barajados de fábrica,
«permutar el orden no cambia las predicciones» sería trivialmente cierto y no diría nada.

La verificación es discriminativa y sin tolerancia absoluta: el vector de cada archivo tiene que
parecerse a su propia fila del tensor más que a cualquiera de las otras 15.

In [ ]:
# La verificación completa está en el change `clasificacion-orientada-a-video`
# (`verificar_orden_temporal.py`). Acá se comprueba lo esencial sobre el CSV.
import csv
from collections import defaultdict

with (rutas.carpeta("resultados") / "dataset_frames_alineados.csv").open(encoding="utf-8") as f:
    filas = list(csv.DictReader(f))
por_clip = defaultdict(list)
for fila in filas:
    por_clip[fila["clip"]].append(fila)

integros = [c["clip"] for c in clips]
desordenados = [c for c in integros
                if sorted(int(f["orden"]) for f in por_clip[c]) != list(range(16))]
print(f"Clips con órdenes 0..15 completos: {len(integros) - len(desordenados)} de {len(integros)}")
assert not desordenados, f"clips con orden incompleto: {desordenados[:5]}"

# Bloque 2: Características del backbone congelado

## Paso 4 — Extraer y cachear

ResNet-18 preentrenada en ImageNet, **sin entrenar**, sin su capa clasificadora: 512 dimensiones
por frame. El *pooling* adaptativo es lo que permite que el recorte no cuadrado de 224×343 entre
sin reescalarlo ni deformarlo.

**Por qué congelado.** Con 141 clips de 73 ejemplares, descongelarlo pone los parámetros
entrenables en el orden de 10⁷ y el sobreajuste es inmediato. No es una elección de arquitectura
por mérito: es lo que 141 muestras permiten sostener.

**Por qué se cachea.** Congelado, para un frame dado siempre devuelve el mismo vector. Comparar
variantes cuesta segundos en vez de minutos.

In [ ]:
if REEXTRAER:
    orden_clips, X = caracteristicas.extraer(clips)
else:
    try:
        orden_clips, X = caracteristicas.cargar()
        print(f"Características vigentes en caché: {X.shape}")
    except (FileNotFoundError, ValueError) as motivo:
        print(f"{motivo}\nExtrayendo...")
        orden_clips, X = caracteristicas.extraer(clips)

indice = {c: i for i, c in enumerate(orden_clips)}

# Bloque 3: Líneas de base

## Paso 5 — Las tres, sobre los mismos folds

Sin ellas un F1 macro de 0,55 no significa nada: puede ser un modelo que aprendió algo o uno que
descubrió la clase mayoritaria.

La de **identidad** es la menos habitual y la más importante. Mide cuánta señal hay en el
confounder de individuo. Con partición por ejemplar el caballo de test nunca se vio, así que cae
al voto mayoritario y empata con la trivial — y **eso es la demostración de que la partición
funciona**, no un defecto de la medición.

In [ ]:
folds = clasificacion.folds_por_ejemplar(clips)

mayoritaria = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_mayoritaria, folds, "1. Clase mayoritaria"))
identidad = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_por_identidad, folds, "2. Sólo id_caballo, sin mirar la imagen"))
un_frame = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_un_frame, folds, "3. Un frame + voto, sin información temporal",
    vectores=X, indice=indice))

## Paso 6 — Cuánto vale el protocolo: la partición ingenua

La línea de base de identidad no puede contestar *¿la clase está determinada por el individuo?*
bajo un protocolo que impide que la identidad transfiera. Se mide con una partición **ingenua por
clip**, la que haría alguien que no agrupa.

Este número es un **diagnóstico del dataset**. No se reporta como resultado del trabajo.

In [ ]:
ingenuo = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_por_identidad, clasificacion.folds_ingenuos(clips),
    "2b. Sólo id_caballo, partición INGENUA por clip"))

fuga = ingenuo["f1_macro"][0] - identidad["f1_macro"][0]
print(f"\nFuga que la partición por ejemplar elimina: {fuga:+.3f} de F1 macro.")
print("La causa es una propiedad del dataset: 63 de los 73 ejemplares aparecen en UNA sola clase,")
print("así que para ellos saber qué caballo es equivale a saber la clase.")

## Paso 7 — El criterio de aceptación, declarado antes de entrenar

*El modelo se considera útil si su F1 macro medio supera al de las tres líneas de base en al menos
un desvío estándar entre folds.* Anclado en las líneas de base y no en un número redondo.

La más alta es la de un frame. **El criterio queda en 0,691.**

In [ ]:
piso = max(mayoritaria["f1_macro"][0], identidad["f1_macro"][0], un_frame["f1_macro"][0])
recalculado = piso + un_frame["f1_macro"][1]

print(f"Línea de base más alta : {piso:.3f} ± {un_frame['f1_macro'][1]:.3f}")
print(f"Recalculado acá        : {recalculado:.4f}")
print(f"DECLARADO en el change : {clasificacion.CRITERIO_ACEPTACION}")
print()
print("La diferencia es redondeo: el criterio se declaró sumando los valores YA REDONDEADOS que")
print(f"figuran en el change (0,626 + 0,065 = 0,691); sin redondear da {recalculado:.4f}.")
print("Se conserva el DECLARADO, que es el que se escribió antes de entrenar. No cambia el")
print("resultado: 0,626 no alcanza ninguno de los dos.")

# Bloque 4: Cabezas temporales

## Paso 8 — Lo que se probó, y por qué no cambió nada

Se compararon **18 configuraciones**: tres cabezas —promedio, GRU, atención— en dos regímenes de
aumentación y dos de capacidad, con las muestras igualadas mediante ventanas de 8 frames.

La distinción que importa, y que el diseño original no hacía: **el promedio es invariante al
orden**. Si gana, la hipótesis no queda sostenida — sería la conclusión de la línea de base de un
frame, disfrazada de modelo secuencial.

| Cabeza | | Entrenables | F1 macro | Permutando el orden |
|---|---|---:|---:|---:|
| `promedio` | invariante | 33 027 | 0,574 ± 0,099 | 0,574 (+0,000) |
| `atencion` | sensible | 33 603 | **0,632 ± 0,155** | 0,632 (+0,000) |
| `gru` | sensible | 42 339 | 0,585 ± 0,114 | 0,619 (−0,034) |
| `promedio` chica | invariante | 4 131 | 0,587 ± 0,078 | 0,587 (+0,000) |
| `atencion` chica | sensible | 4 203 | 0,580 ± 0,098 | 0,580 (+0,000) |
| `gru` chica | sensible | 8 235 | 0,605 ± 0,072 | 0,613 (−0,008) |

**La atención da predicciones idénticas con los frames barajados.** No es un empate estadístico:
es el mismo número hasta el tercer decimal. El modelo *puede* usar el orden y *no lo usa*. Medido
sobre el modelo entrenado: el embedding de posición aporta el 3,5 % de la representación y los
pesos de atención por posición son casi uniformes — **aprendió a promediar**.

**La aumentación no movió nada** (−0,005 a +0,000), con 20 vistas por clip que llevan el
entrenamiento de 106 a 2 120 muestras. Confirma que es un regularizador y no una mejora del
dataset: veinte vistas del mismo clip siguen siendo el mismo caballo.

El código de esta comparación está en el change, en `cabezas_temporales.py`. No se repite acá
porque su resultado es negativo y el modelo que se adopta es el del Bloque 5.

# Bloque 5: Modelo final y evaluación

## Paso 9 — Entrenamiento por fold

**El modelo elegido es el clasificador de un frame con voto** — el que se diseñó como línea de
base. Ganó a las 18 configuraciones y es el más simple y el más estable de todas. Elegir otro
porque «es el que corresponde a un trabajo sobre video» sería elegir contra la evidencia.

In [ ]:
final = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_un_frame, folds, "MODELO FINAL — un frame + voto",
    pesos_en=clasificacion.ruta_pesos, vectores=X, indice=indice))

print(f"\nPesos por fold en: {rutas.carpeta('modelos') / 'objetivo3'}")

## Paso 10 — Contra el criterio declarado

In [ ]:
cumple = clasificacion.contra_el_criterio(final)

## Paso 11 — Matriz de confusión

Sobre las predicciones **fuera de muestra** agregadas de los 4 folds.

In [ ]:
clasificacion.matriz_de_confusion(final["predicciones"])

## Paso 12 — Ablación por `origen_alineacion`

Sólo los clips cuyos 16 frames se alinearon por **detección propia** del modelo de pose, sin
interpolación ni anotación manual. Mismos folds, para que sea comparable.

Se espera que rinda peor: tiene menos material. Lo que mide es **cuánto del resultado se apoya en
posiciones estimadas**.

In [ ]:
detectados = [c for c in clips if c["solo_detectado"]]
from collections import Counter
print("Reparto del subconjunto: " + "  ".join(
    f"{k} {v}" for k, v in sorted(Counter(c["clase"] for c in detectados).items())))

ablacion = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_un_frame,
    clasificacion.folds_por_ejemplar(detectados),
    f"ABLACIÓN — sólo detección propia ({len(detectados)} clips)",
    vectores=X, indice=indice))

print(f"\nDiferencia contra el completo: "
      f"{ablacion['f1_macro'][0] - final['f1_macro'][0]:+.3f} de F1 macro.")
print("El recall de Disgustado cae porque el subconjunto tiene 7 clips de esa clase en 4 folds.")
print("No invalida el resultado completo: es un argumento A FAVOR del relleno temporal y de la")
print("anotación manual del objetivo 2.")

## Paso 13 — Errores por ejemplar

Si los fallos se concentran en pocos caballos, eso es información sobre el **dataset**, no sobre
el modelo.

In [ ]:
clasificacion.errores_por_ejemplar(final["predicciones"])

# Bloque 6: Conclusión

## Paso 14 — Qué quedó demostrado y qué no

**La hipótesis del trabajo no queda sostenida por esta evidencia.** Tres pruebas independientes
dan lo mismo: la cabeza invariante al orden no se distingue de las sensibles al orden; permutar
los frames no cambia las predicciones; y el mejor modelo es invariante al orden por construcción.

**Qué NO dice este resultado.** No dice que la información temporal sea inútil en el problema.
Dice que **con 141 clips de 73 ejemplares no se pudo extraer señal temporal utilizable**.

**Lo que sí quedó demostrado, y es el resultado positivo del trabajo:**

- **Las imágenes tienen señal.** De 0,188 a 0,626 con partición honesta por ejemplar. El
  preprocesamiento facial del objetivo 2 —recorte, alineación por el eje ojo-hocico, encuadre que
  incluye las orejas— produce una entrada de la que un clasificador lineal extrae bastante más que
  las clases a priori.
- **El confounder de individuo está cuantificado.** Sin agrupar por caballo se llega a 0,53 sin
  mirar un píxel.
- **El resultado es honesto por construcción:** criterio declarado antes de entrenar, sin *early
  stopping* sobre el fold de test, 18 configuraciones informadas y no sólo la mejor.

## Qué haría falta para mejorarlo

**Más ejemplares**, y está medido:

1. **Más caballos, no más clips del mismo caballo.** La aumentación se probó y no movió nada.
2. **Más material de `Disgustado` con detección propia.** La ablación lo muestra: con 7 clips la
   clase colapsa.
3. **Revisar las etiquetas de los ejemplares que fallan siempre** y las fronteras entre tomas
   contiguas de un mismo caballo con clases opuestas —el techo del caso `178`—.

Lo que **no** haría falta: más arquitectura.